In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn import metrics

In [2]:
np.random.seed(42)

r2_score_list = []
rmse_score_list = []
for i in range(10):
    data = pd.read_csv('data/data_DFT_mordredpca.csv')
    data['Alc_ID'] = data.index // 14
    shuffled_groups = data['Alc_ID'].unique()
    np.random.shuffle(shuffled_groups)
    train_groups = shuffled_groups[:10]
    test_groups = shuffled_groups[10:]
    train_data = data[data['Alc_ID'].isin(train_groups)].reset_index(drop=True)
    test_data = data[data['Alc_ID'].isin(test_groups)].reset_index(drop=True)

    y_train = pd.DataFrame(train_data['Yield'],columns=['Yield'])
    X_train = train_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    y_test = pd.DataFrame(test_data['Yield'],columns=['Yield'])
    X_test = test_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    
    a_X_train = (X_train - X_train.mean()) / X_train.std()
    a_X_test = (X_test - X_train.mean()) / X_train.std()
    a_X_train = a_X_train.dropna(how='any', axis=1)
    a_X_test = a_X_test[a_X_train.columns]
    param = {'C':[1e+04, 1e+05, 1e+06, 1e+07, 1e+08, 1e+09],
             'gamma':[1e-09, 1e-08, 1e-07, 1e-06, 1e-05, 1e-04],
             'epsilon':[0.1, 1, 10, 50, 100]}
    reg = GridSearchCV(SVR(kernel='rbf'), param_grid=param, cv=5, n_jobs=12)
    reg.fit(a_X_train, y_train['Yield'])
    best = reg.best_estimator_
    #print(f'Run{i} model:', best)
    y_pred1 = best.predict(a_X_train)
    y_pred2 = best.predict(a_X_test)
    r2 = metrics.r2_score(y_test, y_pred2)
    rmse = metrics.root_mean_squared_error(y_test, y_pred2)
    print(f'Run{i} R2 (test):', r2, ', RMSE (test):', rmse)
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print('Mean R2:', np.mean(r2_score_list))
print('SD R2:', np.std(r2_score_list))
print('Mean RMSE:', np.mean(rmse_score_list))
print('SD RMSE:', np.std(rmse_score_list))

Run0 R2 (test): 0.13515244211405086 , RMSE (test): 10.271753503557434
Run1 R2 (test): -0.4028137589537377 , RMSE (test): 35.095870605057755
Run2 R2 (test): -0.28244339946207186 , RMSE (test): 34.71147725778633
Run3 R2 (test): -0.24982926500325808 , RMSE (test): 28.977301498278
Run4 R2 (test): 0.06616459702390454 , RMSE (test): 11.204295766603332
Run5 R2 (test): -0.0026790688919804406 , RMSE (test): 26.90286583986854
Run6 R2 (test): 0.08452129856559198 , RMSE (test): 20.73146411494717
Run7 R2 (test): -0.219225022591506 , RMSE (test): 24.43733806953666
Run8 R2 (test): -0.029895000521380055 , RMSE (test): 19.065100514129377
Run9 R2 (test): -0.09269550210795074 , RMSE (test): 21.4802335177
==========(Result)==========
Mean R2: -0.09937426798283375
SD R2: 0.17118298692057374
Mean RMSE: 23.28777006874646
SD RMSE: 8.122520429294877
